In [0]:
from datetime import datetime
import time

from pyspark.sql.functions import current_timestamp, to_date

#User pass date 
try:
	arrival_date=dbutils.widgets.get("arrival_date")
except Exception:
	arrival_date=datetime.now().strftime("%Y-%m-%d")

#User pass catalog
try:
	catalog=dbutils.widgets.get("catalog")
except Exception:
	catalog="travel_bookings"

#User pass schema
try:
	schema=dbutils.widgets.get("schema")
except Exception:
	schema="default"

#User pass volume
try:
	base_volume=dbutils.widgets.get("base volume")
except Exception:
	base_volume=f"/Volumes/{catalog}/{schema}/data"

In [0]:
#Create booking and customer path
booking_path=f"{base_volume}/booking_data/bookings_{arrival_date}.csv"
customer_path=f"{base_volume}/customer_data/customers_{arrival_date}.csv"

missing=[]

try:
	dbutils.fs.ls(customer_path)
except Exception:
	missing.append(customer_path)

try:
	dbutils.fs.ls(booking_path)
except Exception:
	missing.append(booking_path)

if missing:
	raise FileNotFoundError(f"Missing input file: {missing}")

In [0]:
#Create table and write data
spark.sql(f"create schema if not exists {catalog}.ops")

spark.sql(f"""create table if not exists {catalog}.ops.run_log
(
run_id string, arrival_date date, stage string, status string, message string,
recorded_at timestamp
) using delta
""")

run_id=f"nb-validate-{arrival_date}-{time.time()}"

log_df=spark.createDataFrame([(run_id, datetime.strptime(arrival_date, "%Y-%m-%d").date(), "validate_inputs", "STARTED", "Inputs validate")],
["run_id", "arrival_date", "stage", "status", "message"])

log_df.withColumn("recorded_at", current_timestamp()).write.mode("append").saveAsTable(f"{catalog}.ops.run_log")

print(f"Validation successfull: {booking_path}, {customer_path}")